In [1]:
import os
import gc
import glob
import time
import math
import warnings

# --- 1. ENVIRONMENT CONFIGURATION (Must be first) ---
# Prevents memory fragmentation errors
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore") # Ignore harmless HF warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import GPT2Model, GPT2Tokenizer
from tqdm.auto import tqdm

# Force cleanup of previous runs
gc.collect()
torch.cuda.empty_cache()

print("DEBUG: Environment configured. Memory cleared.")

# --- 2. HYPERPARAMETERS (Optimized for 2x T4 GPUs) ---
BATCH_SIZE = 8           # Total batch size (4 per GPU)
BLOCK_SIZE = 128         # Sequence length (Safe for VRAM)
NEURON_FACTOR = 1        # Multiplier for BDH neuron expansion (1 = Minimal VRAM)
LEARNING_RATE = 2e-4
EPOCHS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count()

print(f"DEBUG: Setup detected {GPU_COUNT} GPUs. Using device: {DEVICE}")
if GPU_COUNT > 1:
    print(f"DEBUG: Dual-GPU mode enabled. Batch Size {BATCH_SIZE} will be split (4 per GPU).")

# --- 3. ROBUST DATA LOADING ---
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def load_data_robust():
    print("DEBUG: Searching for dataset files in /kaggle/input...")
    files = glob.glob('/kaggle/input/**/*.txt', recursive=True)
    
    # Filter for your specific files
    target_files = [f for f in files if "Count of Monte Cristo" in f or "castaways" in f]
    
    if not target_files:
        print("WARNING: Specific novels not found. Generating DUMMY data for code verification.")
        # Create dummy data so the code doesn't crash if files are missing
        dummy_text = "The quick brown fox jumps over the lazy dog. " * 2000
        tokens = tokenizer.encode(dummy_text)
    else:
        print(f"DEBUG: Found {len(target_files)} files: {[os.path.basename(f) for f in target_files]}")
        full_text = ""
        for path in target_files:
            try:
                with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    full_text += content + tokenizer.eos_token
                    print(f"DEBUG: Loaded {len(content)} chars from {os.path.basename(path)}")
            except Exception as e:
                print(f"ERROR loading {path}: {e}")
                
        tokens = tokenizer.encode(full_text)
        print(f"DEBUG: Total tokens encoded: {len(tokens)}")
        
        # Safety Truncation: Don't overload RAM
        if len(tokens) > 1000000:
            print("DEBUG: Truncating dataset to 1M tokens for speed...")
            tokens = tokens[:1000000]

    # Create Blocks
    examples = []
    for i in range(0, len(tokens) - BLOCK_SIZE, BLOCK_SIZE):
        examples.append(tokens[i:i+BLOCK_SIZE])
    
    # Hard cap for rapid prototyping (optional, remove for full training)
    if len(examples) > 10000:
        examples = examples[:10000]
        print("DEBUG: Limiting to 10k samples for this run.")
        
    return torch.tensor(examples)

train_tensor = load_data_robust()
# drop_last=True ensures equal batches for DataParallel
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"DEBUG: DataLoader ready. Batches per epoch: {len(train_loader)}")

# --- 4. BDH MODEL ARCHITECTURE ---
class BDH_LinearAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_neurons_factor=1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.n_neurons = d_model * n_neurons_factor 
        
        # Projections
        self.q_proj = nn.Linear(d_model, self.n_neurons, bias=False)
        self.k_proj = nn.Linear(d_model, self.n_neurons, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.scale = 1.0 / math.sqrt(self.d_head)

    def forward(self, x, return_state=False):
        B, T, D = x.shape
        H = self.n_heads
        
        # Q, K, V Generation
        q = self.q_proj(x).view(B, T, H, -1).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, -1).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, -1).transpose(1, 2)
        
        # BDH Nonlinearity (Sparsity)
        q = F.relu(q)
        k = F.relu(k)
        
        # Recurrent State Calculation (The Dragon's Memory)
        # Uses Einstein Summation for clarity
        kv = torch.einsum('bhtn, bhtd -> bhtnd', k, v)
        state_history = torch.cumsum(kv, dim=2)
        
        # Attention Output
        out = torch.einsum('bhtn, bhtnd -> bhtd', q, state_history) * self.scale
        out = out.transpose(1, 2).reshape(B, T, D)
        out = self.out_proj(out)
        
        if return_state:
            return out, state_history[:, :, -1, :, :]
        return out

class BDH_Adapter_Layer(nn.Module):
    """Wraps BDH Attention to be compatible with GPT-2 Architecture"""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, eps=config.layer_norm_epsilon)
        self.ln_2 = nn.LayerNorm(config.n_embd, eps=config.layer_norm_epsilon)
        # Global NEURON_FACTOR used here
        self.attn = BDH_LinearAttention(config.n_embd, config.n_head, NEURON_FACTOR)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.resid_pdrop),
        )

    def forward(self, hidden_states, *args, **kwargs):
        # **kwargs catches all extra GPT-2 arguments (masks, caches) and ignores them
        return_state = kwargs.get('return_state', False)
        
        x = hidden_states
        residual = x
        x_norm = self.ln_1(x)
        
        if return_state:
            attn_out, final_state = self.attn(x_norm, return_state=True)
        else:
            attn_out = self.attn(x_norm)
            final_state = None
            
        x = residual + attn_out
        residual = x
        x = self.ln_2(x)
        x = self.mlp(x)
        x = residual + x
        
        # Return tuple: (hidden_states, cache/state)
        # We return None for the cache to simulate empty standard cache
        if return_state: return (x, final_state)
        return (x,)

class GraftedBDHModel(nn.Module):
    def __init__(self):
        super().__init__()
        print("DEBUG: Loading base GPT-2 model...")
        self.base_model = GPT2Model.from_pretrained('gpt2')
        
        # ENABLE GRADIENT CHECKPOINTING (Saves massive VRAM)
        self.base_model.gradient_checkpointing_enable()
        print("DEBUG: Gradient Checkpointing Enabled.")
        
        config = self.base_model.config
        
        # SURGERY: Replace standard attention layers with BDH layers
        print("DEBUG: Grafting BDH layers into backbone...")
        new_layers = nn.ModuleList([BDH_Adapter_Layer(config) for _ in range(config.n_layer)])
        self.base_model.h = new_layers
        
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Tie weights
        self.lm_head.weight = self.base_model.wte.weight

    def forward(self, idx, return_last_state=False):
        if return_last_state:
            # Inference mode with State Extraction
            hidden = self.base_model.wte(idx)
            pos = torch.arange(0, idx.size(1), device=idx.device).unsqueeze(0)
            hidden = hidden + self.base_model.wpe(pos)
            
            last_state = None
            for i, layer in enumerate(self.base_model.h):
                if i == len(self.base_model.h) - 1:
                    res = layer(hidden, return_state=True)
                    hidden, last_state = res[0], res[1]
                else:
                    hidden = layer(hidden)[0]
            
            logits = self.lm_head(self.base_model.ln_f(hidden))
            return logits, last_state
        else:
            # Standard Training Forward
            out = self.base_model(idx)
            return self.lm_head(out.last_hidden_state)

# --- 5. INITIALIZATION ---
print("DEBUG: Initializing Grafted Model...")
model = GraftedBDHModel()
model = model.to(DEVICE) # Move to primary GPU

# Apply DataParallel if multiple GPUs available
if GPU_COUNT > 1:
    print(f"DEBUG: Wrapping model in DataParallel for {GPU_COUNT} GPUs.")
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler() # Mixed Precision Scaler

# --- 6. TRAINING LOOP ---
print("\n" + "="*30)
print(f"🚀 STARTING TRAINING (Epochs: {EPOCHS})")
print("="*30 + "\n")

start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    # TQDM for progress estimation
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")
    
    for batch in progress_bar:
        # Get data (DataParallel handles moving to secondary GPUs automatically)
        inputs = batch[0].to(DEVICE)
        
        optimizer.zero_grad()
        
        # Mixed Precision Context
        with torch.cuda.amp.autocast():
            logits = model(inputs)
            
            # Shift logits/labels for causal LM loss
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = inputs[..., 1:].contiguous()
            
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        # Backward Pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        # Update progress bar
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
    
    avg_loss = total_loss / len(train_loader)
    print(f"DEBUG: Epoch {epoch+1} Complete. Avg Loss: {avg_loss:.4f}")

total_duration = time.time() - start_time
print(f"\n✅ TRAINING COMPLETE. Total Time: {total_duration/60:.2f} minutes.")

# --- 7. SAVING ---
print("DEBUG: Saving model...")
save_path = "grafted_bdh_final.pth"

# Handle DataParallel unwrapping
if isinstance(model, nn.DataParallel):
    torch.save(model.module.state_dict(), save_path)
else:
    torch.save(model.state_dict(), save_path)

print(f"DEBUG: Model saved to {save_path}")
print("DEBUG: Ready for Track B Consistency Checking.")

2026-01-08 11:16:36.278211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767870996.301371     892 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767870996.308857     892 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Epoch 1/2:   0%|          | 0/941 [00:00<?, ?batch/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


DEBUG: Epoch 1 Complete. Avg Loss: 5.3408


Epoch 2/2:   0%|          | 0/941 [00:00<?, ?batch/s]

DEBUG: Epoch 2 Complete. Avg Loss: 4.3436

✅ TRAINING COMPLETE. Total Time: 9.69 minutes.
DEBUG: Saving model...
DEBUG: Model saved to grafted_bdh_final.pth
DEBUG: Ready for Track B Consistency Checking.


In [3]:
import torch
import torch.nn.functional as F
import glob
import os

# --- 1. SETUP & DEFINITIONS ---
# Ensure device is defined locally
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def get_novel_path(partial_name):
    """Helper to find files in Kaggle input directory"""
    files = glob.glob('/kaggle/input/**/*.txt', recursive=True)
    for f in files:
        if partial_name in f:
            return f
    return None

# --- 2. CONSISTENCY LOGIC ---
def check_consistency(model, tokenizer, backstory_text, novel_context_tokens=None):
    """
    Calculates perplexity of the backstory.
    Optionally prepends novel context (the 'State' proxy) to see if it surprises the model.
    """
    model.eval()
    
    # Encode backstory
    backstory_tokens = tokenizer.encode(backstory_text, return_tensors='pt').to(device)
    
    # (Optional) If we want to condition on the end of the novel:
    if novel_context_tokens is not None:
        # Concatenate: [Novel Context] + [Backstory]
        input_ids = torch.cat([novel_context_tokens, backstory_tokens], dim=1)
        # We only want to calculate loss on the backstory part, not the context
        target_ids = input_ids.clone()
        target_ids[:, :novel_context_tokens.size(1)] = -100 # Ignore context in loss
    else:
        input_ids = backstory_tokens
        target_ids = input_ids.clone()
    
    with torch.no_grad():
        # Handle DataParallel wrapper if present
        if isinstance(model, torch.nn.DataParallel):
            # DataParallel might split small inputs unevenly, so we use .module for inference on single sample
            logits = model.module(input_ids)
        else:
            logits = model(input_ids)
            
        # Standard Causal Language Modeling Loss
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = target_ids[..., 1:].contiguous()
        
        loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
    return loss.item()

# --- 3. RUN THE TEST ---

# Locate the file dynamically
novel_path = get_novel_path("Count of Monte Cristo")

if novel_path:
    print(f"Reading from: {novel_path}")
    with open(novel_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    
    # Extract the very end of the book to serve as 'context'
    # The BDH/Transformer hybrid will use this to set its internal state
    context_str = text[-2000:] 
    context_tokens = tokenizer.encode(context_str, return_tensors='pt').to(device)
else:
    print("WARNING: Novel file not found. Running in 'Zero-Shot' mode (no context).")
    context_tokens = None

# Test Cases
print("\n--- CONSISTENCY CHECK RESULTS ---")

# 1. Consistent Fact (True to the book)
fact_consistent = "Edmond Dantes escapes from the Chateau d'If by hiding in a burial sack."

# 2. Inconsistent Fact (Contradiction)
fact_inconsistent = "Edmond Dantes escapes from the Chateau d'If by calling an Uber helicopter."

# Calculate "Surprise" (Loss)
# We pass the model and tokenizer explicitly
score_1 = check_consistency(model, tokenizer, fact_consistent, context_tokens)
score_2 = check_consistency(model, tokenizer, fact_inconsistent, context_tokens)

print(f"Scenario 1 (Canonical): Loss = {score_1:.4f}")
print(f"Scenario 2 (Absurd):    Loss = {score_2:.4f}")

delta = score_2 - score_1
print(f"\nDelta (Inconsistent - Consistent): {delta:.4f}")

if delta > 0:
    print("✅ SUCCESS: The model found the inconsistent story more 'surprising' (Higher Loss).")
else:
    print("❌ FAILURE: The model was confused (or hasn't learned the book well enough yet).")

Using device: cuda
Reading from: /kaggle/input/dataset/The Count of Monte Cristo.txt

--- CONSISTENCY CHECK RESULTS ---
Scenario 1 (Canonical): Loss = 8.8922
Scenario 2 (Absurd):    Loss = 9.0288

Delta (Inconsistent - Consistent): 0.1366
✅ SUCCESS: The model found the inconsistent story more 'surprising' (Higher Loss).


In [4]:
# --- COMPLEX CONSISTENCY CHECK: The Fate of Mercedes ---

# 1. Consistent (The Tragic Truth)
# This is complex because it involves three entities: Mercedes, Fernand, and the title 'Countess de Morcerf'.
# The model must accept that Dantes' lover married his enemy.
fact_consistent_complex = (
    "Years after Edmond was taken away to the dungeon, Mercedes, believing him to be dead "
    "and overwhelmed by grief, eventually gave in to the persistent courtship of her cousin Fernand. "
    "She married him and became the Countess de Morcerf, bearing him a son named Albert."
)

# 2. Inconsistent (The 'Happy' Alternate Reality)
# This sounds plausible on the surface (faithful lover), but it fundamentally contradicts 
# the novel's reality (where she marries Fernand). A model storing the 'Book State' should find this surprising.
fact_inconsistent_complex = (
    "Throughout Edmond's long imprisonment, Mercedes remained entirely faithful, refusing to ever marry. "
    "She lived a solitary life as a poor seamstress in the Catalan village, rejecting Fernand's advances "
    "until the day Edmond finally returned to her."
)

print("\n--- COMPLEX NARRATIVE TEST: Mercedes & Fernand ---")

# We reuse the 'check_consistency' and 'context_tokens' from your previous cell
# (Make sure to run the previous cell first so 'model' and 'context_tokens' are loaded!)

score_complex_1 = check_consistency(model, tokenizer, fact_consistent_complex, context_tokens)
score_complex_2 = check_consistency(model, tokenizer, fact_inconsistent_complex, context_tokens)

print(f"Scenario 1 (Canonical/Tragic): Loss = {score_complex_1:.4f}")
print(f"Scenario 2 (False/Faithful):   Loss = {score_complex_2:.4f}")

delta_complex = score_complex_2 - score_complex_1
print(f"\nDelta: {delta_complex:.4f}")

if delta_complex > 0:
    print("✅ SUCCESS: The model knows Mercedes married Fernand (The consistent tragedy is 'less surprising').")
else:
    print("❌ FAILURE: The model prefers the 'faithful lover' trope over the book's actual plot.")


--- COMPLEX NARRATIVE TEST: Mercedes & Fernand ---
Scenario 1 (Canonical/Tragic): Loss = 8.5133
Scenario 2 (False/Faithful):   Loss = 9.2471

Delta: 0.7338
✅ SUCCESS: The model knows Mercedes married Fernand (The consistent tragedy is 'less surprising').
